# Stats 292 — Homework 3
## Word Vector Geometry: Countries, Capitals, and Development

**Due:** 05/07/2026  
**Submit:** Upload your completed `.ipynb` and a rendered `.html` to Canvas.

## Overview

In Lecture 08 we saw that word vector arithmetic encodes semantic structure:

$$v(\text{King}) - v(\text{Man}) + v(\text{Woman}) \approx v(\text{Queen})$$

and that this extends to geographic relationships:

$$v(\text{Capital}(C_1)) - v(\text{Country}(C_1)) \approx v(\text{Capital}(C_2)) - v(\text{Country}(C_2))$$

This homework tests that claim directly. For ~40 countries we will:

1. Retrieve pretrained FastText vectors for each country name and capital name
2. Compute the *capital-minus-country* difference vector $v(\Delta_i) = v(\text{capital}_i) - v(\text{country}_i)$
3. Stack all three sets of vectors into a combined matrix $V$, compute the Gram matrix $V^\top V$, and examine the scree plot
4. Project to 2D via truncated SVD and draw country→capital arrows

Two renderings are produced:

- **Plot A** — color by **continent**
- **Plot B** — color by **GDP per capita quartile** (a proxy for standard of living)

**Notebook references:** Notebook A (SVD rendering pattern); Notebook C (FastText loading pattern, `cos_sim` helper).

## Packages

In [ ]:
# Run these lines ONCE in a terminal (with the stats292 env active) to install
# the two packages not available on conda-forge:
#
#   Rscript -e "install.packages('qs2')"
#   Rscript -e "install.packages('remotes'); remotes::install_gitlab('culturalcartography/text2map.pretrained')"
#
# All other packages (tidyverse, scales) are already installed via conda.
message("R library path: ", .libPaths()[1])

In [2]:
# If any package below is missing, run these lines once in the RStudio R console:
#   install.packages(c("remotes", "qs2", "text2map", "tidyverse", "scales"))
#   remotes::install_gitlab("culturalcartography/text2map.pretrained")

library(qs2)
library(text2map.pretrained)
library(tidyverse)
library(scales)

ERROR: Error in library(qs2): there is no package called ‘qs2’


## Country–Capital Dataset

The dataset contains 42 countries spanning all inhabited continents and a
wide range of per-capita income levels (approximate 2023 World Bank USD).

For **multi-word names** (e.g., "Buenos Aires", "New Delhi") the
`cap_tok` column gives the token(s) to look up in the embedding matrix;
multi-word tokens are averaged component-wise (see helper below).

In [ ]:
df_countries <- tribble(
  ~country,       ~capital,         ~ctry_tok,      ~cap_tok,            ~continent,       ~gdp_pc,
  # Europe — high income
  "Norway",       "Oslo",           "norway",        "oslo",              "Europe",          82500,
  "Switzerland",  "Bern",           "switzerland",   "bern",              "Europe",          85000,
  "Denmark",      "Copenhagen",     "denmark",       "copenhagen",        "Europe",          67000,
  "Australia",    "Canberra",       "australia",     "canberra",          "Oceania",         65000,
  "Canada",       "Ottawa",         "canada",        "ottawa",            "North America",   52000,
  "Israel",       "Jerusalem",      "israel",        "jerusalem",         "Asia",            52000,
  "Sweden",       "Stockholm",      "sweden",        "stockholm",         "Europe",          55000,
  "Netherlands",  "Amsterdam",      "netherlands",   "amsterdam",         "Europe",          55000,
  "Austria",      "Vienna",         "austria",       "vienna",            "Europe",          51000,
  "Germany",      "Berlin",         "germany",       "berlin",            "Europe",          51000,
  "Finland",      "Helsinki",       "finland",       "helsinki",          "Europe",          50000,
  "Belgium",      "Brussels",       "belgium",       "brussels",          "Europe",          47000,
  "France",       "Paris",          "france",        "paris",             "Europe",          43000,
  "Japan",        "Tokyo",          "japan",         "tokyo",             "Asia",            34000,
  "Italy",        "Rome",           "italy",         "rome",              "Europe",          33000,
  "Spain",        "Madrid",         "spain",         "madrid",            "Europe",          30000,
  # Europe / Asia — upper-middle income
  "Portugal",     "Lisbon",         "portugal",      "lisbon",            "Europe",          24000,
  "Greece",       "Athens",         "greece",        "athens",            "Europe",          20000,
  "Poland",       "Warsaw",         "poland",        "warsaw",            "Europe",          18000,
  "Argentina",    "Buenos Aires",   "argentina",     "buenos aires",      "South America",   13000,
  "China",        "Beijing",        "china",         "beijing",           "Asia",            12700,
  "Russia",       "Moscow",         "russia",        "moscow",            "Europe",          12000,
  "Mexico",       "Mexico City",    "mexico",        "mexico city",       "North America",   11000,
  "Turkey",       "Ankara",         "turkey",        "ankara",            "Asia",            10700,
  "Brazil",       "Brasilia",       "brazil",        "brasilia",          "South America",    9700,
  "Thailand",     "Bangkok",        "thailand",      "bangkok",           "Asia",             7700,
  "Colombia",     "Bogota",         "colombia",      "bogota",            "South America",    7300,
  "Peru",         "Lima",           "peru",          "lima",              "South America",    7100,
  "Iran",         "Tehran",         "iran",          "tehran",            "Asia",             7000,
  "Chile",        "Santiago",       "chile",         "santiago",          "South America",   16000,
  # Lower-middle income
  "Indonesia",    "Jakarta",        "indonesia",     "jakarta",           "Asia",             4900,
  "Ukraine",      "Kyiv",           "ukraine",       "kyiv",              "Europe",           4500,
  "Vietnam",      "Hanoi",          "vietnam",       "hanoi",             "Asia",             4300,
  "Morocco",      "Rabat",          "morocco",       "rabat",             "Africa",           3800,
  "Egypt",        "Cairo",          "egypt",         "cairo",             "Africa",           3600,
  # Low income
  "Bangladesh",   "Dhaka",          "bangladesh",    "dhaka",             "Asia",             2600,
  "India",        "New Delhi",      "india",         "delhi",             "Asia",             2500,
  "Ghana",        "Accra",          "ghana",         "accra",             "Africa",           2300,
  "Nigeria",      "Abuja",          "nigeria",       "abuja",             "Africa",           2200,
  "Kenya",        "Nairobi",        "kenya",         "nairobi",           "Africa",           2100,
  "Pakistan",     "Islamabad",      "pakistan",      "islamabad",         "Asia",             1700,
  "Ethiopia",     "Addis Ababa",    "ethiopia",      "addis",             "Africa",           1000,
)

df_countries <- df_countries |>
  mutate(gdp_quartile = paste0("Q", ntile(gdp_pc, 4)),
         gdp_label    = case_when(
           gdp_quartile == "Q1" ~ "Q1 Low (<$4k)",
           gdp_quartile == "Q2" ~ "Q2 Lower-mid ($4-11k)",
           gdp_quartile == "Q3" ~ "Q3 Upper-mid ($11-50k)",
           gdp_quartile == "Q4" ~ "Q4 High (>$50k)"
         ))

cat("Countries:", nrow(df_countries), "\n")
cat("Continents:", paste(sort(unique(df_countries$continent)), collapse = ", "), "\n")
cat("GDP range: $", min(df_countries$gdp_pc), "-", max(df_countries$gdp_pc), "\n")

**Question 1.** Examine the dataset. Which continent has the most countries in
the Q4 (high-income) group? Which has the most in Q1 (low-income)?

## Load Pretrained FastText Embeddings

In [ ]:
pretrained_path <- file.path(
  system.file("data", package = "text2map.pretrained"),
  "vecs_fasttext300_commoncrawl.qs2"
)

if (!file.exists(pretrained_path)) {
  message("Downloading fastText Common Crawl vectors (~1 GB) - once only ...")
  download_pretrained("vecs_fasttext300_commoncrawl")
}

wv <- qs_read(pretrained_path)
cat("Embedding matrix:", nrow(wv), "words x", ncol(wv), "dimensions\n")

## Helper: Retrieve a Word Vector

Country and capital names that are more than one word (e.g., "Buenos Aires")
are handled by averaging the vectors of their constituent tokens.

In [ ]:
# Returns a 300-dim numeric vector, or NULL if no tokens found.
get_vec <- function(token_str, wv) {
  tokens  <- str_split(str_to_lower(str_trim(token_str)), "\\s+")[[1]]
  present <- tokens[tokens %in% rownames(wv)]
  if (length(present) == 0L) return(NULL)
  if (length(present) < length(tokens))
    message("Partial vocab match for '", token_str, "': using ",
            paste(present, collapse = " + "))
  colMeans(wv[present, , drop = FALSE])
}

## Build the Embedding Matrix V

For each country $i$ we retrieve:
- $v_{\text{country},i}$ — vector for the country name
- $v_{\text{capital},i}$ — vector for the capital name
- $v_{\Delta,i} = v_{\text{capital},i} - v_{\text{country},i}$ — the *capital-minus-country* direction

The combined matrix $V$ has $3N$ rows and 300 columns.

In [ ]:
results <- df_countries |>
  rowwise() |>
  mutate(
    vec_ctry = list(get_vec(ctry_tok, wv)),
    vec_cap  = list(get_vec(cap_tok,  wv))
  ) |>
  ungroup() |>
  filter(!map_lgl(vec_ctry, is.null),
         !map_lgl(vec_cap,  is.null)) |>
  mutate(vec_diff = map2(vec_cap, vec_ctry, `-`))

n_valid <- nrow(results)
cat("Countries retained (both tokens found):", n_valid, "\n")

missing <- setdiff(df_countries$country, results$country)
if (length(missing) > 0)
  cat("Dropped (token not in vocab):", paste(missing, collapse = ", "), "\n")

**Question 2.** Which countries or capitals, if any, were NOT found in the
fastText Common Crawl vocabulary?  The Common Crawl was scraped from the
public web in English.  Why might some proper nouns be absent?

In [ ]:
mat_ctry <- do.call(rbind, results$vec_ctry)
mat_cap  <- do.call(rbind, results$vec_cap)
mat_diff <- do.call(rbind, results$vec_diff)

rownames(mat_ctry) <- results$country
rownames(mat_cap)  <- results$capital
rownames(mat_diff) <- paste0(results$country, "_delta")

V <- rbind(mat_ctry, mat_cap, mat_diff)   # 3N x 300

cat("V dimensions:", nrow(V), "x", ncol(V), "\n")

## Gram Matrix and Eigenanalysis

The **Gram matrix** $G = V^\top V$ is a $300 \times 300$ positive
semi-definite matrix.  Its eigenvalues $\lambda_1 \geq \lambda_2 \geq \cdots$
measure how much variance is captured along each spectral direction.

Because $V$ has only $3N \approx 126$ rows, the rank of $G$ is at most
$3N - 1$, so most eigenvalues will be (approximately) zero.

In [ ]:
V_c <- scale(V, center = TRUE, scale = FALSE)

G <- crossprod(V_c)          # V'V, 300 x 300
cat("Gram matrix: 300 x 300\n")

eig <- eigen(G, symmetric = TRUE)

total_var <- sum(eig$values[eig$values > 0])
pct_var   <- pmax(0, eig$values) / total_var

cat(sprintf("Top-5 eigenvalues explain %.1f%% of variance\n", 100 * sum(pct_var[1:5])))
cat(sprintf("Top-2 eigenvalues explain %.1f%% of variance\n", 100 * sum(pct_var[1:2])))

**Question 3.** The 3N embedding vectors live in $\mathbb{R}^{300}$, yet only
a handful of eigenvalues are substantial.  What does this tell you about the
intrinsic dimensionality of the country/capital subspace within the fastText
embedding space?

## Scree Plot

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)

data.frame(k = seq_along(pct_var), pct = 100 * pct_var) |>
  filter(k <= 40, pct > 0) |>
  ggplot(aes(k, pct)) +
    geom_col(fill = "steelblue") +
    geom_line(colour = "firebrick", linewidth = 0.7) +
    geom_point(colour = "firebrick", size = 1.5) +
    labs(x = "Eigenvalue rank k",
         y = "% variance explained",
         title = "Scree plot - Gram matrix of V (country, capital, delta vectors)") +
    theme_minimal()

**Question 4.** Where is the "elbow" in the scree plot?  Mark it.  How many
dimensions would you retain if you were applying the Thurstone rank-selection
criterion (noise floor)?

## Truncated SVD — 2D Rendering

Following the double-SVD approach from Lecture 07, we project $V$ (centered)
onto its first two left singular vectors and scale by the singular values.

In [ ]:
# PCA on country+capital only; including delta rows in the centering biases the
# global mean toward capital vectors ((2/3)*mean_cap instead of the balanced
# (mean_ctry + mean_cap)/2), causing systematic country-above-capital offsets.
V_vis   <- rbind(mat_ctry, mat_cap)        # 2N x 300
pca_res <- prcomp(V_vis, center = TRUE, scale. = FALSE)

# Project delta vectors onto the same axes as supplementary observations
diff_proj <- predict(pca_res, mat_diff)

coords_2d <- rbind(
  as.data.frame(pca_res$x[,   1:2]),
  as.data.frame(diff_proj[,   1:2])
)
colnames(coords_2d) <- c("PC1", "PC2")

type_vec  <- c(rep("Country", n_valid), rep("Capital", n_valid), rep("CapMCity", n_valid))
label_vec <- c(results$country, results$capital, paste0(results$country, " delta"))

df_coords <- coords_2d |>
  mutate(
    type    = type_vec,
    label   = label_vec,
    country = rep(results$country, 3)
  ) |>
  left_join(results |> select(country, continent, gdp_pc, gdp_quartile, gdp_label),
            by = "country")

cat("Variance explained by PC1:", round(100 * pca_res$sdev[1]^2 /
      sum(pca_res$sdev^2), 1), "%\n")
cat("Variance explained by PC2:", round(100 * pca_res$sdev[2]^2 /
      sum(pca_res$sdev^2), 1), "%\n")

In [ ]:
country_pts <- df_coords |>
  filter(type == "Country") |>
  select(country, x0 = PC1, y0 = PC2, continent, gdp_quartile, gdp_label)

capital_pts <- df_coords |>
  filter(type == "Capital") |>
  select(country, x1 = PC1, y1 = PC2)

arrows_df <- left_join(country_pts, capital_pts, by = "country")

## Plot A — Colored by Continent

Each country and its capital are shown as points; an arrow runs from country
to capital.  The capital-minus-country difference vectors (Δ) are shown as
smaller triangles.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 9)

continent_colors <- c(
  "Africa"        = "#E07B39",
  "Asia"          = "#C94040",
  "Europe"        = "#3B6DB3",
  "North America" = "#6BAE45",
  "South America" = "#2AABAB"
)

keep_cont <- names(continent_colors)

points_ac <- df_coords |> filter(type != "CapMCity", continent %in% keep_cont)
delta_ac  <- df_coords |> filter(type == "CapMCity",  continent %in% keep_cont)
labels_ac <- df_coords |> filter(type == "Country",   continent %in% keep_cont)
arrows_ac <- arrows_df |> filter(continent %in% keep_cont)

ggplot() +
  geom_segment(
    data = arrows_ac,
    aes(x = x0, y = y0, xend = x1, yend = y1, colour = continent),
    arrow     = arrow(length = unit(0.15, "cm"), type = "closed"),
    linewidth = 0.5, alpha = 0.7
  ) +
  geom_point(
    data = points_ac,
    aes(PC1, PC2, colour = continent, shape = type),
    size = 2.5
  ) +
  geom_point(
    data = delta_ac,
    aes(PC1, PC2, colour = continent),
    shape = 17, size = 1.2, alpha = 0.6
  ) +
  geom_text(
    data = labels_ac,
    aes(PC1, PC2, label = label, colour = continent),
    size = 2.5, vjust = -0.6, show.legend = FALSE
  ) +
  scale_colour_manual(values = continent_colors, guide = "none") +
  scale_shape_manual(values = c("Country" = 16, "Capital" = 15), name = "Point type") +
  facet_wrap(~ continent, scales = "free", ncol = 3) +
  labs(
    title    = "Word vector geometry: countries and capitals",
    subtitle = "Arrows: country to capital  .  Triangles: delta = capital - country",
    x = "PC 1", y = "PC 2"
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "right")

**Question 5.** Do countries from the same continent cluster together in the
2D rendering?  Which continents are most clearly separated?  Which are most
mixed?

**Question 6.** Examine the **Δ vectors** (triangles) for a single continent.
Do they point in a consistent direction, as the parallelogram analogy predicts?
Are any Δ vectors far from the bulk of their continent's cluster?

## Plot A2 — Difference Vectors Only (All Continents)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 7)

continent_colors_all <- c(
  "Africa"        = "#E07B39",
  "Asia"          = "#C94040",
  "Europe"        = "#3B6DB3",
  "North America" = "#6BAE45",
  "Oceania"       = "#9B59B6",
  "South America" = "#2AABAB"
)

delta_all <- df_coords |> filter(type == "CapMCity")

ggplot(delta_all, aes(PC1, PC2, colour = continent)) +
  geom_point(shape = 17, size = 3) +
  geom_text(aes(label = country), size = 2.5, vjust = -0.6, show.legend = FALSE) +
  scale_colour_manual(values = continent_colors_all, name = "Continent") +
  labs(
    title    = "Capital-minus-country difference vectors (delta)",
    subtitle = "One triangle per country . projected onto the same PC axes as Plot A",
    x = "PC 1", y = "PC 2"
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "right")

**Question 6b.** In Plot A2 all Δ vectors are shown together.  Do triangles from
the same continent cluster together?  Is there a common direction shared across
continents (supporting the parallelogram hypothesis), or do different continents
point in different directions?

## Dimensionality of the Δ Vectors

If the parallelogram property held *perfectly*, every Δ vector would be proportional
(all capital-minus-country differences would point in exactly the same direction),
and `mat_diff` would have rank 1 — one dominant eigenvalue, all others zero.
In practice we expect near-rank-1 structure: one large eigenvalue capturing the
shared "capital-of" direction, and the remainder reflecting noise and
country-specific deviations.

In [ ]:
V_md   <- scale(mat_diff, center = TRUE, scale = FALSE)
G_md   <- crossprod(V_md)
eig_md <- eigen(G_md, symmetric = TRUE)

pct_md <- pmax(0, eig_md$values) / sum(pmax(0, eig_md$values))
cat(sprintf("Top-1 eigenvalue explains %.1f%% of delta variance\n", 100 * pct_md[1]))
cat(sprintf("Top-3 eigenvalues explain %.1f%% of delta variance\n", 100 * sum(pct_md[1:3])))

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)

data.frame(k = 1:40, log_val = log(eig_md$values[1:40])) |>
  ggplot(aes(k, log_val)) +
    geom_point(colour = "steelblue", size = 1.5) +
    geom_line(colour = "steelblue", linewidth = 0.5) +
    labs(x = "Eigenvalue rank k", y = "log(eigenvalue)",
         title = "Log-scree plot - Gram matrix of delta = capital - country vectors") +
    theme_minimal()

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)

data.frame(val = eig_md$values[1:40]) |>
  ggplot(aes(val)) +
    geom_histogram(bins = 25, fill = "steelblue", colour = "white") +
    labs(x = "Eigenvalue", y = "Count",
         title = "Eigenvalue histogram - Gram matrix of delta vectors") +
    theme_minimal()

**Question 7.** In a statistics-naive take, a rank-1 model should have one
dimension account for the bulk of the variance in the Δ vectors.  According to
this take, is the structure closer to rank-1 (one dominant eigenvalue, sharp
drop), or is significant variance spread across many dimensions?

**Question 8.** Let's pursue the sophisticated-statistics take.  The histogram
shows the distribution of the top 50 eigenvalues.  A statistician will discern a
"bulk + outlier" structure: the noise is vastly larger than the 1-d signal, yet
the 1-d structure is still clearly present.  What might the statistician say about
how universal the capital-minus-country relationship is across different languages
and cultures represented in Common Crawl?

## Plot B — Colored by GDP per Capita Quartile

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 8)

gdp_colors <- c(
  "Q1 Low (<$4k)"          = "#D73027",
  "Q2 Lower-mid ($4-11k)"  = "#FC8D59",
  "Q3 Upper-mid ($11-50k)" = "#91BFDB",
  "Q4 High (>$50k)"        = "#1A75BA"
)

gdp_order <- c("Q1 Low (<$4k)", "Q2 Lower-mid ($4-11k)",
               "Q3 Upper-mid ($11-50k)", "Q4 High (>$50k)")

arrows_gdp <- arrows_df |>
  mutate(gdp_label = factor(gdp_label, levels = gdp_order))

points_gdp <- df_coords |>
  filter(type != "CapMCity") |>
  mutate(gdp_label = factor(gdp_label, levels = gdp_order))

ggplot() +
  geom_segment(
    data = arrows_gdp,
    aes(x = x0, y = y0, xend = x1, yend = y1, colour = gdp_label),
    arrow     = arrow(length = unit(0.15, "cm"), type = "closed"),
    linewidth = 0.5, alpha = 0.7
  ) +
  geom_point(
    data = points_gdp,
    aes(PC1, PC2, colour = gdp_label, shape = type),
    size = 2.5
  ) +
  geom_point(
    data = df_coords |> filter(type == "CapMCity") |>
             mutate(gdp_label = factor(gdp_label, levels = gdp_order)),
    aes(PC1, PC2, colour = gdp_label),
    shape = 17, size = 1.2, alpha = 0.6
  ) +
  geom_text(
    data = df_coords |> filter(type == "Country") |>
             mutate(gdp_label = factor(gdp_label, levels = gdp_order)),
    aes(PC1, PC2, label = label, colour = gdp_label),
    size = 2.5, vjust = -0.6, show.legend = FALSE
  ) +
  scale_colour_manual(values = gdp_colors, name = "GDP/capita", drop = FALSE) +
  scale_shape_manual(values = c("Country" = 16, "Capital" = 15), name = "Point type") +
  labs(
    title    = "Word vector geometry: countries and capitals",
    subtitle = "Color by GDP per capita quartile . Arrows: country to capital . Triangles: delta",
    x        = "PC 1", y = "PC 2"
  ) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "right")

**Question 9.** Do high-income (Q4) countries cluster separately from
low-income (Q1) countries?  If so, does the separation run primarily along
PC1, PC2, or both?

**Question 10.** GDP per capita correlates strongly with continent (most Q4
countries are European, most Q1 are African or South Asian).  Can you find
any country that is a clear **outlier** — e.g., a high-income country that
clusters near low-income countries in the embedding, or vice versa?  What
might explain this in terms of how the word appears in web text?

## Conceptual Questions

**Question 11.**  The word "paris" encodes associations from all its uses in
the Common Crawl training corpus: fashion, literature, geography, the 2015
climate accords, etc.  How might these non-geographic associations distort
the 2D plot?  Would you expect a corpus restricted to geographic texts to
produce a cleaner clustering?

**Question 12.**  The difference vector $\Delta_i = v(\text{capital}_i) -
v(\text{country}_i)$ is intended to capture the semantic relationship
"is-the-capital-of."  Name one other semantic relationship you could test
with the same method (e.g., using anchor words other than capital cities).
Write the analogy equation and describe what you would expect to see in a 2D
rendering.

**Question 13 (Extension).**  The **Orthogonal Procrustes** method
(Lecture 07) aligns two embedding matrices.  Design an experiment in which
you:
(a) train two separate embeddings — one on pre-2010 web text, one on
post-2020 web text;
(b) Procrustes-align them;
(c) measure how much the vectors for individual country names have moved.

Which countries do you predict would show the largest shift?  Why?

In [ ]:
sessionInfo()